In [1]:
import pandas as pd
import numpy as np

sol = pd.read_csv("FinAcceso_solicitudes_credito.csv")
mer = pd.read_csv("FinAcceso_mercado_potencial.csv")

In [2]:
sol["decision"].value_counts()

decision
Rechazada    662
Aprobada     658
Name: count, dtype: int64

In [10]:
sol["aprobado"] = (sol["decision"] == "Aprobada").astype(int)
sol["aprobado"].mean() * 100


np.float64(49.848484848484844)

La tasa global de aprobación es 49,8 %

In [15]:
t3 = sol.groupby("zona").agg(
    solicitudes=("aprobado", "count"),
    aprobaciones=("aprobado", "sum"),
    tasa_aprob=("aprobado", "mean")
)
t3["tasa_aprob"] = (t3["tasa_aprob"] * 100).round(1)
t3.sort_values("aprobaciones", ascending=False)

,solicitudes,aprobaciones,tasa_aprob
zona,,,
Norte,475,295,62.1
Centro,395,193,48.9
Sur,318,129,40.6
Periferia,132,41,31.1


In [16]:
t3.sort_values("tasa_aprob", ascending=False)

,solicitudes,aprobaciones,tasa_aprob
zona,,,
Norte,475,295,62.1
Centro,395,193,48.9
Sur,318,129,40.6
Periferia,132,41,31.1


-El orden de las zonas no cambia entre números absolutos y tasas

-Sería engañoso interpretar que Norte tiene más demanda o que allí la asesoría rinde más. Ni las aprobaciones ni la tasa miden necesidad la tasa refleja quienes lo solicitan.

-Antes de comparar tasas entre zonas conviene revisar el puntaje de riesgo, porque si los solicitantes de Periferia tienen puntajes más bajos, su menor tasa se explicaría por el puntaje y no por la zona, y ese puntaje heredado podría reflejar desigualdades previas.

In [19]:
sol.groupby("zona")["puntaje_riesgo"].median()

zona
Centro       659.0
Norte        670.0
Periferia    594.0
Sur          614.5
Name: puntaje_riesgo, dtype: float64

In [20]:
sol["historial_crediticio"].value_counts()

historial_crediticio
Amplio      824
Limitado    496
Name: count, dtype: int64

In [21]:
def rango_puntaje(p):
    if p < 610:
        return "<610"
    elif p < 690:
        return "610-689"
    else:
        return "690+"

sol["rango_puntaje"] = sol["puntaje_riesgo"].apply(rango_puntaje)
sol["rango_puntaje"].value_counts()

rango_puntaje
610-689    490
<610       448
690+       382
Name: count, dtype: int64

In [22]:
t4 = sol.groupby(["rango_puntaje", "historial_crediticio"]).agg(
    solicitudes=("decision", "count"),
    pct_aprob=("decision", lambda x: (x == "Aprobada").mean() * 100)
)
t4.round(1)

solicitudes  pct_aprob
rango_puntaje historial_crediticio                        
610-689       Amplio                        319       58.3
              Limitado                      171       44.4
690+          Amplio                        318       82.7
              Limitado                       64       73.4
<610          Amplio                        187       24.6
              Limitado                      261       15.3